# 🧠 Spatial-Symbolic Transformer for ARC

## Overview
This notebook implements a **position-aware transformer architecture** specifically designed for ARC matrix reasoning:
- **Input**: Matrix values with explicit (x, y) positions
- **Architecture**: Spatial-Symbolic Transformer with self-attention
- **Goal**: Native spatial reasoning for discrete symbolic matrices

## Key Innovation
Unlike CNN-based encoders that treat matrices as images, this architecture:
- Converts matrices to sequences of (value, x, y) tokens
- Uses self-attention to learn spatial relationships
- Handles variable matrix sizes naturally
- Provides explicit positional reasoning capabilities

In [ ]:
# Import required libraries
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import math
import time
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# Set matplotlib style
plt.style.use('default')
sns.set_palette("husl")

print("\n🚀 Spatial-Symbolic Transformer Environment Ready!")

In [ ]:
# Load ARC dataset (same as before)
def load_arc_data():
    """Load ARC training data from Kaggle input directory"""
    
    # Kaggle data paths
    train_challenges_path = '/kaggle/input/arc-prize-2025/arc-agi_training_challenges.json'
    train_solutions_path = '/kaggle/input/arc-prize-2025/arc-agi_training_solutions.json'
    
    try:
        # Load training challenges
        with open(train_challenges_path, 'r') as f:
            challenges = json.load(f)
        
        # Load training solutions
        with open(train_solutions_path, 'r') as f:
            solutions = json.load(f)
        
        print(f"✅ Loaded {len(challenges)} training tasks")
        print(f"✅ Loaded {len(solutions)} training solutions")
        
        return challenges, solutions
    
    except FileNotFoundError as e:
        print(f"❌ ARC data files not found: {e}")
        print("📁 Expected paths:")
        print(f"   • {train_challenges_path}")
        print(f"   • {train_solutions_path}")
        print("\n💡 Solutions:")
        print("   1. Upload this notebook to Kaggle with ARC Prize 2025 dataset")
        print("   2. Or modify the paths above to point to your local ARC data files")
        print("   3. Download ARC data from: https://www.kaggle.com/competitions/arc-prize-2025/data")
        
        raise FileNotFoundError("ARC dataset not found. Please check file paths or run on Kaggle.")

# Load the data
print("🔄 Loading ARC dataset...")
challenges, solutions = load_arc_data()

# Quick dataset verification
sample_task_id = list(challenges.keys())[0]
sample_task = challenges[sample_task_id]

print(f"\n📋 Sample Task ID: {sample_task_id}")
print(f"📊 Number of training examples: {len(sample_task['train'])}")
print(f"🧪 Number of test examples: {len(sample_task['test'])}")

# Show first training example structure
first_example = sample_task['train'][0]
input_grid = np.array(first_example['input'])
output_grid = np.array(first_example['output'])

print(f"\n🔍 First training example analysis:")
print(f"   Input shape: {input_grid.shape}")
print(f"   Output shape: {output_grid.shape}")
print(f"   Input unique values: {np.unique(input_grid)}")
print(f"   Output unique values: {np.unique(output_grid)}")

print(f"\n✅ Data loading successful! Ready for spatial-symbolic processing.")

In [ ]:
# Step 3: Matrix-to-Sequence Converter - Core Innovation
class MatrixToSequenceConverter:
    """
    Converts ARC matrices to sequences of (value, x, y) tokens for transformer processing.
    This is the key innovation that makes position explicit rather than implicit.
    """
    
    def __init__(self, normalize_positions=True):
        self.normalize_positions = normalize_positions
    
    def matrix_to_sequence(self, matrix):
        """
        Convert matrix to sequence of position-aware tokens.
        
        Args:
            matrix: numpy array of shape (height, width) with values 0-9
            
        Returns:
            tokens: list of (value, x, y) tuples
            metadata: dict with original dimensions and stats
        """
        height, width = matrix.shape
        tokens = []
        
        # Convert each cell to (value, x, y) token
        for y in range(height):
            for x in range(width):
                value = int(matrix[y, x])
                
                if self.normalize_positions:
                    # Normalize positions to [0, 1] range for better learning
                    norm_x = x / max(width - 1, 1)  # Avoid division by zero
                    norm_y = y / max(height - 1, 1)
                    tokens.append((value, norm_x, norm_y))
                else:
                    # Use absolute positions
                    tokens.append((value, x, y))
        
        # Metadata for reconstruction and analysis
        metadata = {
            'height': height,
            'width': width,
            'num_tokens': len(tokens),
            'unique_values': len(np.unique(matrix)),
            'total_cells': height * width
        }
        
        return tokens, metadata
    
    def sequence_to_matrix(self, tokens, target_height, target_width):
        """
        Convert sequence back to matrix (for reconstruction testing).
        
        Args:
            tokens: list of (value, x, y) tuples
            target_height, target_width: desired output dimensions
            
        Returns:
            matrix: reconstructed numpy array
        """
        matrix = np.zeros((target_height, target_width), dtype=int)
        
        for value, x, y in tokens:
            if self.normalize_positions:
                # Denormalize positions
                abs_x = int(round(x * max(target_width - 1, 1)))
                abs_y = int(round(y * max(target_height - 1, 1)))
            else:
                abs_x, abs_y = int(x), int(y)
            
            # Ensure coordinates are within bounds
            if 0 <= abs_y < target_height and 0 <= abs_x < target_width:
                matrix[abs_y, abs_x] = int(value)
        
        return matrix
    
    def visualize_conversion(self, matrix, tokens, metadata):
        """Visualize the matrix-to-sequence conversion process."""
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Original matrix
        im1 = ax1.imshow(matrix, cmap='tab10', vmin=0, vmax=9)
        ax1.set_title(f'Original Matrix ({metadata["height"]}x{metadata["width"]})')
        ax1.set_xlabel('X coordinate')
        ax1.set_ylabel('Y coordinate')
        
        # Add grid and labels
        ax1.set_xticks(range(metadata['width']))
        ax1.set_yticks(range(metadata['height']))
        ax1.grid(True, alpha=0.3)
        
        # Token sequence visualization
        values = [token[0] for token in tokens]
        x_coords = [token[1] for token in tokens]
        y_coords = [token[2] for token in tokens]
        
        scatter = ax2.scatter(x_coords, y_coords, c=values, cmap='tab10', 
                            vmin=0, vmax=9, s=100, alpha=0.8)
        ax2.set_title(f'Token Sequence ({len(tokens)} tokens)')
        ax2.set_xlabel('Normalized X' if self.normalize_positions else 'X coordinate')
        ax2.set_ylabel('Normalized Y' if self.normalize_positions else 'Y coordinate')
        
        # Add colorbar
        plt.colorbar(scatter, ax=ax2, label='Value')
        
        plt.tight_layout()
        plt.show()
        
        return fig

# Test the Matrix-to-Sequence Converter
print("🔧 Testing Matrix-to-Sequence Conversion...")

# Initialize converter
converter = MatrixToSequenceConverter(normalize_positions=True)

# Test with the sample matrix we loaded
test_matrix = input_grid  # Use the first example from ARC data
print(f"\n📊 Testing with matrix shape: {test_matrix.shape}")
print(f"Matrix content:\n{test_matrix}")

# Convert to sequence
tokens, metadata = converter.matrix_to_sequence(test_matrix)

print(f"\n🎯 Conversion Results:")
print(f"   • Original matrix: {metadata['height']}x{metadata['width']} = {metadata['total_cells']} cells")
print(f"   • Token sequence: {metadata['num_tokens']} tokens")
print(f"   • Unique values: {metadata['unique_values']}")

# Show first few tokens
print(f"\n🔍 First 10 tokens (value, norm_x, norm_y):")
for i, token in enumerate(tokens[:10]):
    value, x, y = token
    print(f"   Token {i}: value={value}, x={x:.3f}, y={y:.3f}")

# Test reconstruction
reconstructed = converter.sequence_to_matrix(tokens, metadata['height'], metadata['width'])
reconstruction_accurate = np.array_equal(test_matrix, reconstructed)

print(f"\n✅ Reconstruction Test:")
print(f"   • Original == Reconstructed: {reconstruction_accurate}")
if reconstruction_accurate:
    print("   🎉 Perfect round-trip conversion!")
else:
    print("   ❌ Reconstruction error - need to debug")

# Visualize the conversion
print(f"\n📈 Visualizing conversion process...")
converter.visualize_conversion(test_matrix, tokens, metadata)

In [ ]:
# Step 4: Position-Value Joint Embedding
class PositionValueEmbedding(nn.Module):
    """
    Creates joint embeddings for (value, x, y) triplets.
    This combines discrete value embeddings with continuous position embeddings.
    """
    
    def __init__(self, embed_dim=128, max_seq_length=900):  # 30x30 = 900 max tokens
        super().__init__()
        self.embed_dim = embed_dim
        self.max_seq_length = max_seq_length
        
        # Value embedding: discrete values 0-9 → dense vectors
        self.value_embedding = nn.Embedding(10, embed_dim // 2)  # Half the dimensions
        
        # Position embeddings: continuous (x, y) → dense vectors  
        self.position_mlp = nn.Sequential(
            nn.Linear(2, embed_dim // 4),  # (x, y) → quarter dimensions
            nn.ReLU(),
            nn.Linear(embed_dim // 4, embed_dim // 2)  # → half dimensions
        )
        
        # Final projection to combine value + position
        self.final_projection = nn.Linear(embed_dim, embed_dim)
        
        # Layer normalization for stable training
        self.layer_norm = nn.LayerNorm(embed_dim)
        
        print(f"✅ PositionValueEmbedding initialized:")
        print(f"   • Embedding dimension: {embed_dim}")
        print(f"   • Value embedding: 10 → {embed_dim // 2}D")
        print(f"   • Position embedding: 2 → {embed_dim // 2}D") 
        print(f"   • Combined output: {embed_dim}D")
    
    def forward(self, token_sequence):
        """
        Embed sequence of (value, x, y) tokens.
        
        Args:
            token_sequence: tensor of shape (batch_size, seq_len, 3)
                          where each token is [value, x, y]
        
        Returns:
            embeddings: tensor of shape (batch_size, seq_len, embed_dim)
        """
        batch_size, seq_len, _ = token_sequence.shape
        
        # Extract components
        values = token_sequence[:, :, 0].long()  # (batch_size, seq_len)
        positions = token_sequence[:, :, 1:3]    # (batch_size, seq_len, 2)
        
        # Embed values (discrete)
        value_embeds = self.value_embedding(values)  # (batch_size, seq_len, embed_dim//2)
        
        # Embed positions (continuous)
        pos_embeds = self.position_mlp(positions)   # (batch_size, seq_len, embed_dim//2)
        
        # Concatenate value and position embeddings
        combined = torch.cat([value_embeds, pos_embeds], dim=-1)  # (batch_size, seq_len, embed_dim)
        
        # Final projection and normalization
        embeddings = self.final_projection(combined)
        embeddings = self.layer_norm(embeddings)
        
        return embeddings
    
    def get_embedding_info(self):
        """Return information about the embedding structure."""
        return {
            'total_params': sum(p.numel() for p in self.parameters()),
            'value_embedding_params': sum(p.numel() for p in self.value_embedding.parameters()),
            'position_mlp_params': sum(p.numel() for p in self.position_mlp.parameters()),
            'embed_dim': self.embed_dim
        }

# Test Position-Value Joint Embedding
print("🧠 Testing Position-Value Joint Embedding...")

# Initialize embedding layer
embed_dim = 128
embedding_layer = PositionValueEmbedding(embed_dim=embed_dim).to(device)

# Convert our test tokens to tensor format
def tokens_to_tensor(tokens, batch_size=1):
    """Convert list of (value, x, y) tuples to tensor format."""
    token_tensor = torch.tensor(tokens, dtype=torch.float32)
    
    # Add batch dimension if needed
    if batch_size > 1:
        # For batch processing, we'd need to pad sequences to same length
        # For now, just single batch
        token_tensor = token_tensor.unsqueeze(0)  # (1, seq_len, 3)
    else:
        token_tensor = token_tensor.unsqueeze(0)  # (1, seq_len, 3)
    
    return token_tensor.to(device)

# Test with our converted tokens
test_tokens_tensor = tokens_to_tensor(tokens)
print(f"\n📊 Input tensor shape: {test_tokens_tensor.shape}")
print(f"Input tensor content:\n{test_tokens_tensor}")

# Forward pass through embedding
with torch.no_grad():
    embeddings = embedding_layer(test_tokens_tensor)

print(f"\n🎯 Embedding Results:")
print(f"   • Input shape: {test_tokens_tensor.shape}")
print(f"   • Output shape: {embeddings.shape}")
print(f"   • Embedding dim: {embeddings.shape[-1]}")
print(f"   • Output range: [{embeddings.min():.3f}, {embeddings.max():.3f}]")

# Show embedding info
embed_info = embedding_layer.get_embedding_info()
print(f"\n📈 Embedding Layer Statistics:")
for key, value in embed_info.items():
    if 'params' in key:
        print(f"   • {key}: {value:,}")
    else:
        print(f"   • {key}: {value}")

print(f"\n✅ Position-Value Embedding test successful!")
print(f"🚀 Ready for Step 5: Spatial-Symbolic Transformer Blocks!")

In [ ]:
# Step 5: Spatial-Symbolic Transformer Block
class SpatialSymbolicTransformerBlock(nn.Module):
    """
    Transformer block specifically designed for spatial reasoning with discrete symbols.
    Uses multi-head self-attention to learn spatial relationships between positions.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        
        # Multi-head self-attention for spatial relationships
        self.self_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True  # Input format: (batch, seq, embed_dim)
        )
        
        # Feed-forward network for feature transformation
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, embed_dim),
            nn.Dropout(dropout)
        )
        
        # Layer normalization for residual connections
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
        print(f"✅ SpatialSymbolicTransformerBlock initialized:")
        print(f"   • Embed dim: {embed_dim}")
        print(f"   • Num heads: {num_heads}")
        print(f"   • FF dim: {ff_dim}")
        print(f"   • Dropout: {dropout}")
    
    def forward(self, x, return_attention=False):
        """
        Forward pass through transformer block.
        
        Args:
            x: input embeddings (batch_size, seq_len, embed_dim)
            return_attention: whether to return attention weights
            
        Returns:
            output: transformed embeddings (batch_size, seq_len, embed_dim)
            attention_weights: if return_attention=True
        """
        # Self-attention with residual connection
        attn_output, attn_weights = self.self_attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        if return_attention:
            return x, attn_weights
        return x

class SpatialSymbolicTransformer(nn.Module):
    """
    Complete Spatial-Symbolic Transformer for ARC matrix reasoning.
    Stacks multiple transformer blocks for deep spatial understanding.
    """
    
    def __init__(self, embed_dim=128, num_heads=8, ff_dim=512, num_layers=6, 
                 dropout=0.1, output_dim=1024):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_layers = num_layers
        self.output_dim = output_dim
        
        # Position-Value embedding layer
        self.embedding = PositionValueEmbedding(embed_dim=embed_dim)
        
        # Stack of transformer blocks
        self.transformer_blocks = nn.ModuleList([
            SpatialSymbolicTransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        
        # Global aggregation to fixed output size
        self.global_pool = nn.AdaptiveAvgPool1d(1)  # Pool across sequence dimension
        
        # Final projection to output dimension
        self.output_projection = nn.Sequential(
            nn.Linear(embed_dim, output_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, output_dim)
        )
        
        # Layer normalization for final output
        self.output_norm = nn.LayerNorm(output_dim)
        
        print(f"✅ SpatialSymbolicTransformer initialized:")
        print(f"   • Input: Variable-size matrix → {output_dim}D features")
        print(f"   • Architecture: {num_layers} transformer blocks")
        print(f"   • Total parameters: {sum(p.numel() for p in self.parameters()):,}")
    
    def forward(self, token_sequence, return_attention=False):
        """
        Forward pass: matrix tokens → global features.
        
        Args:
            token_sequence: (batch_size, seq_len, 3) - (value, x, y) tokens
            return_attention: whether to return attention from last layer
            
        Returns:
            global_features: (batch_size, output_dim)
            attention_weights: if return_attention=True
        """
        # Embed tokens: (batch_size, seq_len, 3) → (batch_size, seq_len, embed_dim)
        x = self.embedding(token_sequence)
        
        # Pass through transformer blocks
        attention_weights = None
        for i, block in enumerate(self.transformer_blocks):
            if return_attention and i == len(self.transformer_blocks) - 1:
                x, attention_weights = block(x, return_attention=True)
            else:
                x = block(x)
        
        # Global aggregation: (batch_size, seq_len, embed_dim) → (batch_size, embed_dim)
        # Transpose for adaptive pooling: (batch_size, embed_dim, seq_len)
        x = x.transpose(1, 2)
        x = self.global_pool(x).squeeze(-1)  # (batch_size, embed_dim)
        
        # Final projection: (batch_size, embed_dim) → (batch_size, output_dim)
        global_features = self.output_projection(x)
        global_features = self.output_norm(global_features)
        
        if return_attention:
            return global_features, attention_weights
        return global_features
    
    def get_model_info(self):
        """Get detailed model information."""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'model_size_mb': total_params * 4 / (1024 * 1024),  # 4 bytes per float32
            'embed_dim': self.embed_dim,
            'num_layers': self.num_layers,
            'output_dim': self.output_dim
        }

# Test Spatial-Symbolic Transformer
print("🤖 Testing Complete Spatial-Symbolic Transformer...")

# Initialize the full transformer
transformer = SpatialSymbolicTransformer(
    embed_dim=128,
    num_heads=8,
    ff_dim=512,
    num_layers=6,
    output_dim=1024
).to(device)

# Test with our token sequence
print(f"\n📊 Testing with token sequence shape: {test_tokens_tensor.shape}")

# Forward pass
with torch.no_grad():
    global_features, attention_weights = transformer(test_tokens_tensor, return_attention=True)

print(f"\n🎯 Transformer Results:")
print(f"   • Input shape: {test_tokens_tensor.shape}")
print(f"   • Output shape: {global_features.shape}")
print(f"   • Feature range: [{global_features.min():.3f}, {global_features.max():.3f}]")
print(f"   • Attention shape: {attention_weights.shape if attention_weights is not None else 'None'}")

# Model information
model_info = transformer.get_model_info()
print(f"\n📈 Model Statistics:")
for key, value in model_info.items():
    if 'params' in key:
        print(f"   • {key}: {value:,}")
    elif 'size_mb' in key:
        print(f"   • {key}: {value:.1f} MB")
    else:
        print(f"   • {key}: {value}")

print(f"\n✅ Spatial-Symbolic Transformer test successful!")
print(f"🎉 We now have a complete position-aware encoder that converts matrices → 1024D features!")
print(f"🚀 Ready for Step 6: Dataset integration and reconstruction testing!")